# Silver Layer: Clean and Standardize Data

This notebook transforms the raw bronze data into `silver.csv`. The silver layer keeps the same business grain as the source data, but standardizes names, text values, dates, duplicates, and missing values so the gold model can be built reliably.


## Setup

Import the required libraries and keep input/output file paths as constants.


In [1]:
from pathlib import Path
import pandas as pd

BRONZE_DATA_PATH = Path("Bronze.csv")
SILVER_DATA_PATH = Path("Silver.csv")


## Helper Functions


In [2]:
def add_date_parts(dataframe, timestamp_column="timestamp"):
    """Convert a timestamp column into year, month name, and day name columns."""
    cleaned = dataframe.copy()
    timestamp = pd.to_datetime(cleaned[timestamp_column], errors="coerce")
    return (
        cleaned.assign(
            year=timestamp.dt.year,
            month=timestamp.dt.month_name(),
            day=timestamp.dt.day_name(),
        )
        .drop(columns=timestamp_column)
    )


def fill_missing_values(dataframe):
    """Fill missing values with simple column statistics."""
    cleaned = dataframe.copy()
    # A missing parent means the post is not a reply/share, so 0 is safer than a fake median ID.
    if "parent_post_id" in cleaned.columns:
        cleaned["parent_post_id"] = cleaned["parent_post_id"].fillna(0)
    numeric_fill_values = cleaned.select_dtypes(include="number").median(numeric_only=True)
    text_fill_values = cleaned.select_dtypes(include=["object", "string"]).mode(dropna=True).iloc[0]

    return (
        cleaned.fillna(numeric_fill_values)
        .fillna(text_fill_values)
        .reset_index(drop=True)
    )


## Load Raw Data

Start from the original CSV and preview the first rows before changing anything.


In [3]:
df_raw = pd.read_csv(BRONZE_DATA_PATH)
df_raw.head()


,post_id,platform,timestamp,crisis_phase,topic,sentiment_label,sentiment_score,emotion,stance_label,claim_type,...,impressions_estimate,reach_estimate,engagement_velocity,misinformation_probability,credibility_score,uncertainty_index,subjectivity_score,toxicity_score,region,language
0,1,Twitter,2025-06-05 18:23:00,Escalation,Service disruption,Neutral,-0.108,Fear,Support,Opinion,...,30897,10254,7.808,0.308,0.712,0.503,0.749,0.036,Global,English
1,2,Reddit,2025-08-11 21:56:00,Recovery,User complaints,Neutral,0.063,Trust,Neutral,Fact Claim,...,34685,17812,20.347,0.110,0.721,0.396,0.585,0.240,Global,English
2,3,Twitter,2025-07-05 02:09:00,Peak,User complaints,Negative,-0.364,Frustration,Question,Fact Claim,...,109789,45897,143.045,0.746,0.105,0.676,0.372,0.452,Global,English
3,4,Reddit,2025-08-27 18:01:00,Response,Privacy concerns,Neutral,0.176,Confusion,Criticize,Observation,...,39077,14778,31.070,0.095,0.737,0.239,0.281,0.022,Global,English
4,5,Reddit,2025-07-10 10:20:00,Recovery,Service disruption,Positive,0.516,Trust,Criticize,Fact Claim,...,71736,52714,21.411,0.147,0.859,0.356,0.479,0.082,Global,English


## Standardize Columns and Text

Column names become consistent `snake_case`, duplicate rows are removed, and text values are normalized so categories match cleanly during grouping and joins.


In [4]:
df_silver = df_raw.drop_duplicates()
df_silver.head()


,post_id,platform,timestamp,crisis_phase,topic,sentiment_label,sentiment_score,emotion,stance_label,claim_type,...,impressions_estimate,reach_estimate,engagement_velocity,misinformation_probability,credibility_score,uncertainty_index,subjectivity_score,toxicity_score,region,language
0,1,Twitter,2025-06-05 18:23:00,Escalation,Service disruption,Neutral,-0.108,Fear,Support,Opinion,...,30897,10254,7.808,0.308,0.712,0.503,0.749,0.036,Global,English
1,2,Reddit,2025-08-11 21:56:00,Recovery,User complaints,Neutral,0.063,Trust,Neutral,Fact Claim,...,34685,17812,20.347,0.110,0.721,0.396,0.585,0.240,Global,English
2,3,Twitter,2025-07-05 02:09:00,Peak,User complaints,Negative,-0.364,Frustration,Question,Fact Claim,...,109789,45897,143.045,0.746,0.105,0.676,0.372,0.452,Global,English
3,4,Reddit,2025-08-27 18:01:00,Response,Privacy concerns,Neutral,0.176,Confusion,Criticize,Observation,...,39077,14778,31.070,0.095,0.737,0.239,0.281,0.022,Global,English
4,5,Reddit,2025-07-10 10:20:00,Recovery,Service disruption,Positive,0.516,Trust,Criticize,Fact Claim,...,71736,52714,21.411,0.147,0.859,0.356,0.479,0.082,Global,English


## Derive Calendar Fields

The raw timestamp is converted into separate year, month, and day fields. These fields support the date dimension in the gold layer.


In [5]:
df_silver = add_date_parts(df_silver)
df_silver[["year", "month", "day"]].head()


,year,month,day
0,2025,June,Thursday
1,2025,August,Monday
2,2025,July,Saturday
3,2025,August,Wednesday
4,2025,July,Thursday


## Fill Missing Values

Missing values are filled after the structural cleaning. Numeric columns use the median, text columns use the mode, and `parent_post_id` uses `0` because missing parents represent original posts.


In [6]:
df_silver = fill_missing_values(df_silver)
df_silver.info()


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 33 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   post_id                     50000 non-null  int64  
 1   platform                    50000 non-null  str    
 2   crisis_phase                50000 non-null  str    
 3   topic                       50000 non-null  str    
 4   sentiment_label             50000 non-null  str    
 5   sentiment_score             50000 non-null  float64
 6   emotion                     50000 non-null  str    
 7   stance_label                50000 non-null  str    
 8   claim_type                  50000 non-null  str    
 9   follower_count              50000 non-null  int64  
 10  following_count             50000 non-null  int64  
 11  account_age_days            50000 non-null  int64  
 12  verified_flag               50000 non-null  int64  
 13  source_type                 50000 non-null

## Validate the Clean Dataset

Preview the final silver dataset before exporting it.


In [7]:
df_silver.head()


,post_id,platform,crisis_phase,topic,sentiment_label,sentiment_score,emotion,stance_label,claim_type,follower_count,...,misinformation_probability,credibility_score,uncertainty_index,subjectivity_score,toxicity_score,region,language,year,month,day
0,1,Twitter,Escalation,Service disruption,Neutral,-0.108,Fear,Support,Opinion,5193,...,0.308,0.712,0.503,0.749,0.036,Global,English,2025,June,Thursday
1,2,Reddit,Recovery,User complaints,Neutral,0.063,Trust,Neutral,Fact Claim,1735,...,0.110,0.721,0.396,0.585,0.240,Global,English,2025,August,Monday
2,3,Twitter,Peak,User complaints,Negative,-0.364,Frustration,Question,Fact Claim,2933,...,0.746,0.105,0.676,0.372,0.452,Global,English,2025,July,Saturday
3,4,Reddit,Response,Privacy concerns,Neutral,0.176,Confusion,Criticize,Observation,891,...,0.095,0.737,0.239,0.281,0.022,Global,English,2025,August,Wednesday
4,5,Reddit,Recovery,Service disruption,Positive,0.516,Trust,Criticize,Fact Claim,4445,...,0.147,0.859,0.356,0.479,0.082,Global,English,2025,July,Thursday


## Export Silver Data

Save the cleaned dataset for the gold modeling notebook.


In [8]:
df_silver.to_csv(SILVER_DATA_PATH, index=False)
print(f"Saved {len(df_silver):,} rows to {SILVER_DATA_PATH}")


Saved 50,000 rows to Silver.csv


In [9]:
df_silver.head()

,post_id,platform,crisis_phase,topic,sentiment_label,sentiment_score,emotion,stance_label,claim_type,follower_count,...,misinformation_probability,credibility_score,uncertainty_index,subjectivity_score,toxicity_score,region,language,year,month,day
0,1,Twitter,Escalation,Service disruption,Neutral,-0.108,Fear,Support,Opinion,5193,...,0.308,0.712,0.503,0.749,0.036,Global,English,2025,June,Thursday
1,2,Reddit,Recovery,User complaints,Neutral,0.063,Trust,Neutral,Fact Claim,1735,...,0.110,0.721,0.396,0.585,0.240,Global,English,2025,August,Monday
2,3,Twitter,Peak,User complaints,Negative,-0.364,Frustration,Question,Fact Claim,2933,...,0.746,0.105,0.676,0.372,0.452,Global,English,2025,July,Saturday
3,4,Reddit,Response,Privacy concerns,Neutral,0.176,Confusion,Criticize,Observation,891,...,0.095,0.737,0.239,0.281,0.022,Global,English,2025,August,Wednesday
4,5,Reddit,Recovery,Service disruption,Positive,0.516,Trust,Criticize,Fact Claim,4445,...,0.147,0.859,0.356,0.479,0.082,Global,English,2025,July,Thursday
